## 导入井头、井分层、井曲线数据


### 1) 环境与路径设置

确保可从仓库根目录导入 `src` 下的模块，并定位原始数据路径。


In [ ]:
import sys
from pathlib import Path

import lasio
import pandas as pd

# 以 notebooks/ 为起点回到仓库根目录
repo_root = Path.cwd().resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from cup.petrel.load import (
    extract_gr_log_from_las,
    extract_rho_log_from_las,
    extract_vp_log_from_las,
    extract_vs_log_from_las,
    import_well_heads_petrel,
    import_well_tops_petrel,
)
from wtie.processing import grid

data_root = repo_root / "data"
well_heads_file = data_root / "raw" / "well_heads"
well_tops_file = data_root / "raw" / "well_tops"
las_dir = data_root / "vertical_well_las_QYZ"

print(f"repo_root: {repo_root}")
print(f"well_heads_file exists: {well_heads_file.exists()}")
print(f"well_tops_file exists: {well_tops_file.exists()}")
print(f"las_dir exists: {las_dir.exists()}")

### 2) 导入井头与井分层文件

使用 `load.py` 中现有函数读取并检查字段。


In [ ]:
well_heads_df = import_well_heads_petrel(well_heads_file)
well_tops_df = import_well_tops_petrel(well_tops_file)

print("well_heads shape:", well_heads_df.shape)
print("well_tops shape:", well_tops_df.shape)

display(well_heads_df.head())
display(well_tops_df.head())

### 3) 批量导入 LAS 并构建含 Vp, Vs, Rho, GR 的 LogSet

遍历 `data/vertical_well_las` 下全部 `.las` 文件，逐井提取曲线并组装。


In [ ]:
las_files = sorted(las_dir.glob("*.las"))
print(f"LAS files found: {len(las_files)}")
for p in las_files:
    print(" -", p.name)

logsets = {}
failed = []

for las_path in las_files:
    try:
        las_file = lasio.read(las_path)

        vp_log = extract_vp_log_from_las(las_file)
        vs_log = extract_vs_log_from_las(las_file)
        rho_log = extract_rho_log_from_las(las_file)
        gr_log = extract_gr_log_from_las(las_file)

        logset = grid.LogSet({"Vp": vp_log, "Rho": rho_log})
        logset.add_log("Vs", vs_log)
        logset.add_log("GR", gr_log)

        # 以文件名（不含后缀）作为井名键
        logsets[las_path.stem] = logset

    except Exception as e:
        failed.append((las_path.name, str(e)))

print(f"\n成功构建 LogSet 数量: {len(logsets)}")
print(f"失败数量: {len(failed)}")
if failed:
    for name, msg in failed:
        print(f"[FAILED] {name}: {msg}")

### 4) 结果检查

检查每口井是否都包含 Vp、Vs、Rho、GR，并给出曲线长度与采样间隔。


In [ ]:
summary_rows = []
for well_name, ls in logsets.items():
    summary_rows.append(
        {
            "well": well_name,
            "logs": list(ls.Logs.keys()),
            "n_samples": ls.Vp.size,
            "sampling_rate": ls.sampling_rate,
            "basis_type": ls.basis_type,
            "has_Vp": "Vp" in ls.Logs,
            "has_Vs": "Vs" in ls.Logs,
            "has_Rho": "Rho" in ls.Logs,
            "has_GR": "GR" in ls.Logs,
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values("well").reset_index(drop=True)
display(summary_df)

if len(summary_df) > 0:
    all_ok = bool((summary_df[["has_Vp", "has_Vs", "has_Rho", "has_GR"]].all(axis=1)).all())
    print("全部井均含 Vp/Vs/Rho/GR:", all_ok)

# 示例查看第一口井的前几行
if logsets:
    first_well = sorted(logsets.keys())[0]
    print("\n示例井:", first_well)
    display(logsets[first_well].df.head())